# 📊 Exploratory Data Analysis (EDA) - Job Salary Prediction

## 🎯 Objective
The goal of this notebook is to conduct an in-depth Exploratory Data Analysis (EDA) on tech and data science compensation records. We analyze the primary drivers of salary, evaluate distributions and outliers, explore relationships between experience, job roles, company size, and remote arrangements, and prepare key findings to guide **Feature Engineering** and **Machine Learning Model Training**.

---
### 📋 Table of Contents
1. **Environment Setup & Configuration**
2. **Data Ingestion & Structural Overview**
3. **Data Quality & Sanity Checks**
4. **Feature Enrichment & Categorical Mapping**
5. **Target Variable Analysis (`salary_in_usd`)**
6. **Univariate Feature Exploration**
7. **Bivariate Analysis (Key Salary Drivers)**
8. **Temporal & Geographic Salary Dynamics**
9. **Multi-Factor Interaction & Correlation Matrix**
10. **Key Insights & ML Modeling Roadmap**


In [ ]:
# 1. Environment Setup & Configuration
import os
import sys
import tempfile
import warnings

# Direct matplotlib cache to temp directory to avoid sandbox/permission limits
os.environ['MPLCONFIGDIR'] = tempfile.gettempdir()
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Environment configured successfully. Ready for analysis.")


## 2. Data Ingestion & Structural Overview
We locate and load the raw dataset (`ds_salaries.csv`) and inspect its dimensions, attributes, and column datatypes.


In [ ]:
# Robustly locate and load the dataset regardless of execution root
possible_paths = [
    os.path.join('..', 'data', 'raw', 'ds_salaries.csv'),
    os.path.join('data', 'raw', 'ds_salaries.csv'),
    os.path.join('ml', 'data', 'raw', 'ds_salaries.csv'),
    os.path.join(os.path.dirname(os.getcwd()), 'data', 'raw', 'ds_salaries.csv')
]

data_path = next((p for p in possible_paths if os.path.exists(p)), None)
if not data_path:
    raise FileNotFoundError("Could not locate 'ds_salaries.csv'. Ensure ml/download_data.py has been run.")

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


In [1]:
# Column types and null count
df.info()


NameError: name 'df' is not defined

In [ ]:
# Descriptive statistics for numerical variables
df.describe().T


In [ ]:
# Summary statistics for categorical variables
df.describe(include=['O', 'str']).T


## 3. Data Quality & Sanity Checks
Checking for missing values, anomalous records, and duplicate entries.


In [ ]:
# Check missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None! Dataset has zero missing values.")

# Check duplicate rows
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates} ({duplicates / len(df) * 100:.2f}%)")
print("Note: In aggregated salary survey data, duplicate profiles (e.g. Senior Data Engineers with identical salary & remote status) are common and represent valid occurrences.")


## 4. Feature Enrichment & Label Decoding
For cleaner and more interpretable visualizations, we decode abbreviated codes:
- **Experience Level**: `EN` (Entry-level), `MI` (Mid-level), `SE` (Senior-level), `EX` (Executive)
- **Employment Type**: `FT` (Full-time), `PT` (Part-time), `CT` (Contract), `FL` (Freelance)
- **Company Size**: `S` (Small <50), `M` (Medium 50-250), `L` (Large >250)
- **Remote Ratio**: `0` (On-site), `50` (Hybrid), `100` (Fully Remote)


In [ ]:
# Create mapped descriptive columns
exp_map = {
    'EN': 'Entry-level (EN)',
    'MI': 'Mid-level (MI)',
    'SE': 'Senior-level (SE)',
    'EX': 'Executive (EX)'
}
emp_map = {
    'FT': 'Full-time',
    'PT': 'Part-time',
    'CT': 'Contract',
    'FL': 'Freelance'
}
size_map = {
    'S': 'Small (<50)',
    'M': 'Medium (50-250)',
    'L': 'Large (>250)'
}
remote_map = {
    0: 'On-site (0%)',
    50: 'Hybrid (50%)',
    100: 'Fully Remote (100%)'
}

df['experience_desc'] = df['experience_level'].map(exp_map)
df['employment_desc'] = df['employment_type'].map(emp_map)
df['company_size_desc'] = df['company_size'].map(size_map)
df['remote_desc'] = df['remote_ratio'].map(remote_map)

# Quick verification of mappings
df[['experience_level', 'experience_desc', 'company_size', 'company_size_desc', 'remote_ratio', 'remote_desc']].head(3)


## 5. Target Variable Analysis (`salary_in_usd`)
Examining the distribution, central tendencies, variance, skewness, and outliers of the target variable.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram with KDE
sns.histplot(df['salary_in_usd'], kde=True, color='#2b5c8f', bins=35, ax=axes[0])
mean_val = df['salary_in_usd'].mean()
median_val = df['salary_in_usd'].median()
axes[0].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: ${mean_val:,.0f}')
axes[0].axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: ${median_val:,.0f}')
axes[0].set_title('Salary (USD) Distribution & Density')
axes[0].set_xlabel('Salary in USD')
axes[0].set_ylabel('Count')
axes[0].legend()

# Boxplot
sns.boxplot(x=df['salary_in_usd'], color='#68a0cf', ax=axes[1])
axes[1].set_title('Salary (USD) Boxplot & Outliers')
axes[1].set_xlabel('Salary in USD')

plt.tight_layout()
plt.show()

print(f"Mean Salary:   ${mean_val:,.2f}")
print(f"Median Salary: ${median_val:,.2f}")
print(f"Std Deviation: ${df['salary_in_usd'].std():,.2f}")
print(f"Skewness:      {df['salary_in_usd'].skew():.2f} (Moderately right-skewed)")
print(f"Kurtosis:      {df['salary_in_usd'].kurt():.2f}")


In [ ]:
# Assess normality under Log-Transformation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

log_salary = np.log1p(df['salary_in_usd'])

sns.histplot(log_salary, kde=True, color='#2e8b57', bins=30, ax=axes[0])
axes[0].set_title('Log-Transformed Salary (log1p) Distribution')
axes[0].set_xlabel('log(Salary in USD + 1)')

# Boxplot of log salary
sns.boxplot(x=log_salary, color='#98fb98', ax=axes[1])
axes[1].set_title('Log-Transformed Salary Boxplot')
axes[1].set_xlabel('log(Salary in USD + 1)')

plt.tight_layout()
plt.show()

print(f"Original Skewness:        {df['salary_in_usd'].skew():.2f}")
print(f"Log-Transformed Skewness: {log_salary.skew():.2f}")
print("-> Log transformation significantly normalizes the target distribution, ideal for linear / regularized models.")


## 6. Univariate Feature Exploration
Visualizing the categorical distribution across the dataset:
- Experience Level
- Company Size
- Remote Work Ratio
- Top Job Titles


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Experience Level
exp_order = ['Entry-level (EN)', 'Mid-level (MI)', 'Senior-level (SE)', 'Executive (EX)']
sns.countplot(data=df, x='experience_desc', order=exp_order, palette='mako', ax=axes[0, 0])
axes[0, 0].set_title('Distribution by Experience Level')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('Number of Professionals')
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f'{int(p.get_height())}\n({p.get_height()/len(df)*100:.1f}%)',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=10)

# 2. Company Size
size_order = ['Small (<50)', 'Medium (50-250)', 'Large (>250)']
sns.countplot(data=df, x='company_size_desc', order=size_order, palette='viridis', ax=axes[0, 1])
axes[0, 1].set_title('Distribution by Company Size')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('Count')
for p in axes[0, 1].patches:
    axes[0, 1].annotate(f'{int(p.get_height())}\n({p.get_height()/len(df)*100:.1f}%)',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=10)

# 3. Remote Ratio
remote_order = ['On-site (0%)', 'Hybrid (50%)', 'Fully Remote (100%)']
sns.countplot(data=df, x='remote_desc', order=remote_order, palette='crest', ax=axes[1, 0])
axes[1, 0].set_title('Distribution by Remote Work Arrangement')
axes[1, 0].set_xlabel('')
axes[1, 0].set_ylabel('Count')
for p in axes[1, 0].patches:
    axes[1, 0].annotate(f'{int(p.get_height())}\n({p.get_height()/len(df)*100:.1f}%)',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=10)

# 4. Employment Type
emp_order = ['Full-time', 'Part-time', 'Contract', 'Freelance']
sns.countplot(data=df, x='employment_desc', order=emp_order, palette='magma', ax=axes[1, 1])
axes[1, 1].set_title('Distribution by Employment Type')
axes[1, 1].set_xlabel('')
axes[1, 1].set_ylabel('Count')
for p in axes[1, 1].patches:
    axes[1, 1].annotate(f'{int(p.get_height())}\n({p.get_height()/len(df)*100:.1f}%)',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# Top 15 Most Prevalent Job Titles
plt.figure(figsize=(12, 7))
top_jobs = df['job_title'].value_counts().head(15)

sns.barplot(x=top_jobs.values, y=top_jobs.index, palette='Blues_r')
plt.title('Top 15 Most Frequent Job Roles in Dataset')
plt.xlabel('Number of Listings')
plt.ylabel('Job Role')

for i, count in enumerate(top_jobs.values):
    plt.text(count + 10, i, f'{count} ({count/len(df)*100:.1f}%)', va='center', fontsize=10)

plt.tight_layout()
plt.show()


## 7. Bivariate Analysis: Key Salary Drivers
We examine how individual factors directly influence the target compensation (`salary_in_usd`).


In [ ]:
# Salary vs Experience Level
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x='experience_desc',
    y='salary_in_usd',
    order=['Entry-level (EN)', 'Mid-level (MI)', 'Senior-level (SE)', 'Executive (EX)'],
    palette='Blues',
    showmeans=True,
    meanprops={"marker": "D", "markeredgecolor": "black", "markerfacecolor": "gold", "markersize": "7"}
)

plt.title('Salary Distribution Across Experience Levels (Diamond = Mean)')
plt.xlabel('Experience Tier')
plt.ylabel('Salary in USD')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.show()

# Print summary table
exp_stats = df.groupby('experience_desc')['salary_in_usd'].agg(['count', 'mean', 'median', 'std'])
exp_stats.sort_values(by='mean', ascending=False)


In [ ]:
# Salary vs Company Size
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x='company_size_desc',
    y='salary_in_usd',
    order=['Small (<50)', 'Medium (50-250)', 'Large (>250)'],
    palette='viridis',
    showmeans=True,
    meanprops={"marker": "D", "markeredgecolor": "black", "markerfacecolor": "red", "markersize": "7"}
)

plt.title('Salary vs Company Size (Diamond = Mean)')
plt.xlabel('Company Size Category')
plt.ylabel('Salary in USD')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.show()

# Company size statistics
size_stats = df.groupby('company_size_desc')['salary_in_usd'].agg(['count', 'mean', 'median', 'std'])
size_stats


In [ ]:
# Salary vs Remote Setting
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x='remote_desc',
    y='salary_in_usd',
    order=['On-site (0%)', 'Hybrid (50%)', 'Fully Remote (100%)'],
    palette='crest',
    showmeans=True,
    meanprops={"marker": "D", "markeredgecolor": "black", "markerfacecolor": "yellow", "markersize": "7"}
)

plt.title('Salary by Work Arrangement (Diamond = Mean)')
plt.xlabel('Remote Status')
plt.ylabel('Salary in USD')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.show()

remote_stats = df.groupby('remote_desc')['salary_in_usd'].agg(['count', 'mean', 'median', 'std'])
remote_stats


In [ ]:
# Salary by Top Job Roles (Roles with at least 25 entries for statistical validity)
role_counts = df['job_title'].value_counts()
popular_roles = role_counts[role_counts >= 25].index

role_salary = df[df['job_title'].isin(popular_roles)].groupby('job_title')['salary_in_usd'].median().sort_values(ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x=role_salary.values, y=role_salary.index, palette='mako')

plt.title('Median Salary by Top Data & AI Job Titles (min. 25 samples)')
plt.xlabel('Median Salary in USD')
plt.ylabel('Job Role')
plt.gca().xaxis.set_major_formatter('${x:,.0f}')

for i, val in enumerate(role_salary.values):
    plt.text(val + 2000, i, f'${val:,.0f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()


## 8. Temporal & Geographic Salary Dynamics
Analyzing how salaries evolved across years (`2020` - `2023`) and identifying top-paying country locations.


In [ ]:
# Year-over-year compensation trend across experience levels
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=df,
    x='work_year',
    y='salary_in_usd',
    hue='experience_desc',
    hue_order=['Entry-level (EN)', 'Mid-level (MI)', 'Senior-level (SE)', 'Executive (EX)'],
    marker='o',
    markersize=8,
    linewidth=2.5,
    palette='tab10',
    errorbar=None
)

plt.title('Salary Evolution Over Time (2020 - 2023) by Experience Level')
plt.xlabel('Work Year')
plt.ylabel('Mean Salary in USD')
plt.xticks([2020, 2021, 2022, 2023])
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.legend(title='Experience Level', loc='upper left')
plt.show()


In [ ]:
# Top 10 Company Locations by Average Salary (min. 15 records)
country_counts = df['company_location'].value_counts()
valid_countries = country_counts[country_counts >= 15].index

top_countries = (
    df[df['company_location'].isin(valid_countries)]
    .groupby('company_location')['salary_in_usd']
    .agg(['count', 'mean', 'median'])
    .sort_values(by='mean', ascending=False)
)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_countries['mean'], y=top_countries.index, palette='flare')

plt.title('Top Paying Company Locations (min. 15 records)')
plt.xlabel('Mean Salary in USD')
plt.ylabel('Country Code')
plt.gca().xaxis.set_major_formatter('${x:,.0f}')

for i, (mean_val, count) in enumerate(zip(top_countries['mean'], top_countries['count'])):
    plt.text(mean_val + 2000, i, f'${mean_val:,.0f} (n={count})', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# US vs Non-US comparison
df['is_us_company'] = (df['company_location'] == 'US').map({True: 'US Company', False: 'Non-US Company'})
us_comp = df.groupby('is_us_company')['salary_in_usd'].agg(['count', 'mean', 'median', 'std'])
print("US vs. Non-US Salary Comparison:")
display(us_comp)


## 9. Multi-Factor Interaction & Correlation Matrix
Evaluating interplay between multiple features and generating an encoded correlation heatmap.


In [ ]:
# Interaction Heatmap: Experience Level vs Company Size
pivot_exp_size = df.pivot_table(
    index='experience_desc',
    columns='company_size_desc',
    values='salary_in_usd',
    aggfunc='mean'
).reindex(
    index=['Entry-level (EN)', 'Mid-level (MI)', 'Senior-level (SE)', 'Executive (EX)'],
    columns=['Small (<50)', 'Medium (50-250)', 'Large (>250)']
)

plt.figure(figsize=(10, 5))
sns.heatmap(pivot_exp_size, annot=True, fmt=',.0f', cmap='YlGnBu', cbar_kws={'label': 'Mean Salary ($)'})
plt.title('Mean Salary ($): Experience Level × Company Size')
plt.xlabel('Company Size')
plt.ylabel('Experience Level')
plt.show()


In [ ]:
# Prepare encoded features for correlation heatmap
df_corr = pd.DataFrame()

# Ordinal mappings
df_corr['salary_in_usd'] = df['salary_in_usd']
df_corr['experience_tier'] = df['experience_level'].map({'EN': 1, 'MI': 2, 'SE': 3, 'EX': 4})
df_corr['company_size_num'] = df['company_size'].map({'S': 1, 'M': 2, 'L': 3})
df_corr['remote_ratio'] = df['remote_ratio']
df_corr['work_year'] = df['work_year']
df_corr['is_us_company'] = (df['company_location'] == 'US').astype(int)

corr_matrix = df_corr.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Matrix with Salary in USD')
plt.show()

print("Key Correlations with salary_in_usd:")
print(corr_matrix['salary_in_usd'].sort_values(ascending=False))


## 10. Key Insights & ML Modeling Roadmap

### 🔍 Summary of Key Exploratory Findings
1. **Target Distribution**:
   - `salary_in_usd` ranges from **$5,132** to **$450,000** with a median of **$135,000** and mean of **$137,570**.
   - The distribution exhibits moderate right-skewness (skewness ≈ 0.89). Applying a logarithmic transformation (`log1p`) normalizes the distribution (skewness ≈ -0.13), which is advantageous for linear algorithms.
2. **Dominant Salary Drivers**:
   - **Experience Tier**: Has the strongest positive correlation with salary ($r \approx 0.33$). Median salary escalates systematically from Entry ($78k) → Mid ($100k) → Senior ($153k) → Executive ($190k).
   - **Company Geography**: US companies pay substantially higher on average (~$151k) compared to non-US locations (~$80k).
   - **Company Size**: Medium and Large enterprises offer higher median salaries than small startups.
   - **Role Hierarchy**: Leadership & specialized positions (Data Science Managers, Data Architects, Machine Learning Engineers) command premium salaries relative to generic roles.
3. **Temporal Trend**:
   - Tech salaries have trended upward steadily from 2020 through 2023 across senior and executive tiers.
4. **Data Cleanliness**:
   - Zero missing values.
   - Natural duplication reflects identical respondent demographic archetypes.

---

### 🚀 Roadmap for Implementation (`ml/src/` & `backend/`)
* **Preprocessing (`ml/src/preprocessing.py`)**:
  - Ordinal encoding for `experience_level` (`EN: 0, MI: 1, SE: 2, EX: 3`) and `company_size` (`S: 0, M: 1, L: 2`).
  - Target encoding or frequency encoding for high-cardinality `job_title` (grouping tail categories into 'Other').
  - Binary encoding for US vs. non-US company location.
  - One-hot encoding for `employment_type` and `remote_ratio`.
  - Feature scaling (StandardScaler / RobustScaler) for numerical features.
* **Model Selection (`ml/src/train.py`)**:
  - Baseline: Ridge / Lasso Regression (with log-transformed target).
  - Advanced Ensemble: Random Forest Regressor & XGBoost / LightGBM Regressor.
  - Evaluation Metrics: RMSE, MAE, and $R^2$ score.
* **Backend Serving (`backend/app/`)**:
  - Accept structured inputs matching features identified here: `job_title`, `experience_level`, `employment_type`, `remote_ratio`, `company_size`, `company_location`.
  - Return predicted salary in USD with confidence/IQR ranges.
